# Segment Wise Modelling

This notebook runs segment-wise **one-week-ahead** tea price forecasting on `tea_preprocessed_v2.csv` for four segments: High Grown, Low Grown, Off-Grade, and Dust.

It creates a next-auction target and evaluates XGBoost, LightGBM, Gradient Boosting, and Random Forest with mandatory time-series cross-validation to avoid data leakage.

Features include engineered columns from `feature_engineering.py`, lagged weather variables (1/2/3-week lags for precipitation, sunshine, and temperature), and structural features (`grade_rank`, `tier_rank`, `prestige_rank`).

In [1]:
"""Imports and global config for segment-wise time-series forecasting."""

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

SEED = 42
np.random.seed(SEED)

try:
    from xgboost import XGBRegressor
except ImportError as exc:
    raise ImportError("Please install xgboost: pip install xgboost") from exc

try:
    from lightgbm import LGBMRegressor
except ImportError as exc:
    raise ImportError("Please install lightgbm: pip install lightgbm") from exc

print("Libraries loaded.")

Libraries loaded.


In [2]:
"""Load data, build one-week-ahead target, validate engineered input, and prepare segment dataframes."""

df_raw = pd.read_csv("../data/Processed/tea_preprocessed_v3.csv")
TARGET_BASE = "price_mid_lkr"
TARGET = "price_mid_lkr_t_plus_1w"
FORECAST_HORIZON = "t_plus_1_week"
SEGMENT_COL = "category_type"

# Keep the original row count for traceability to source dataset size.
raw_rows = len(df_raw)
df = df_raw.copy()

# Basic cleaning for robust model training.
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET_BASE, SEGMENT_COL]).copy()

# Stable chronological ordering before creating t+1 target.
if "sale_year" in df.columns:
    df["_sale_year_num"] = pd.to_numeric(df["sale_year"], errors="coerce")
else:
    df["_sale_year_num"] = np.nan
if "sale_number" in df.columns:
    df["_sale_num"] = pd.to_numeric(df["sale_number"], errors="coerce")
else:
    df["_sale_num"] = np.nan
df["_sale_id_norm"] = pd.to_numeric(df.get("sale_id", np.nan), errors="coerce")

df = df.sort_values(["_sale_year_num", "_sale_num", "_sale_id_norm"], kind="mergesort").reset_index(drop=True)

# One-week-ahead forecast target within each segment stream.
df[TARGET] = df.groupby(SEGMENT_COL, sort=False)[TARGET_BASE].shift(-1)
df = df.dropna(subset=[TARGET]).copy()
df = df.drop(columns=["_sale_year_num", "_sale_num", "_sale_id_norm"] )

# Validate that feature_engineering.py outputs are present in this input.
fe_interact = [c for c in df.columns if c.startswith("interact__")]
fe_roll = [c for c in df.columns if c.startswith("roll3_")]
fe_poly = [c for c in df.columns if c.startswith("poly2__")]
if not (fe_interact and fe_roll and fe_poly):
    raise ValueError(
        "Feature-engineered columns not found. Please generate tea_preprocessed_v2.csv using feature_engineering.py."
    )

# Structural features requested in the methodology.
grade_rank_map = {
    "bop": 4, "bopf": 4, "bop1": 4,
    "fbop": 3, "fbop1": 3, "fbopf": 3, "fbopf1": 3, "fbopf_tippy": 3,
    "op": 2, "op1": 2, "opa": 2,
    "pekoe": 1, "pek1": 1, "pekoe_fbop": 1,
}
tier_rank_map = {"select_best": 4, "best": 3, "below_best": 2, "others": 1}
prestige_rank_map = {"high": 3, "medium": 2, "low": 1}

df["grade_rank"] = df.get("grade", pd.Series("", index=df.index)).astype(str).str.lower().map(grade_rank_map).fillna(0)
df["tier_rank"] = df.get("tier", pd.Series("", index=df.index)).astype(str).str.lower().map(tier_rank_map).fillna(df.get("tier_enc", 0)).fillna(0)
df["prestige_rank"] = df.get("elevation", pd.Series("", index=df.index)).astype(str).str.lower().map(prestige_rank_map).fillna(0)

# Required lagged weather variables: lag1/lag2/lag3 for precipitation, sunshine, temperature.
lag_weather_cols = [
    c for c in df.columns
    if ("lag1" in c or "lag2" in c or "lag3" in c)
    and ("precipitation" in c or "sunshine" in c or "temperature" in c)
]

# Remove target-derived and same-auction price proxy columns to prevent leakage.
drop_cols = {
    TARGET_BASE,
    TARGET,
    "price_mid_lkr_log",
    "price_mid_usd",
    "price_lo_lkr",
    "price_hi_lkr",
    "price_range_lkr",
    "sale_id",
    "sale_date_raw",
    "has_price_target",
}
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
candidate_features = [c for c in numeric_cols if c not in drop_cols]

# Force required methodological features into feature set if present.
required_structural = ["grade_rank", "tier_rank", "prestige_rank"]
required_features = [c for c in (lag_weather_cols + required_structural) if c in df.columns]
feature_cols = sorted(list(set(candidate_features + required_features)))

# Create 4 required segment dataframes.
segments = {
    "high_grown": df[df[SEGMENT_COL] == "high_grown"].copy(),
    "low_grown": df[df[SEGMENT_COL] == "low_grown"].copy(),
    "off_grade": df[df[SEGMENT_COL] == "off_grade"].copy(),
    "dust": df[df[SEGMENT_COL] == "dust"].copy(),
}

segment_labels = {
    "high_grown": "High Grown",
    "low_grown": "Low Grown",
    "off_grade": "Off-Grade",
    "dust": "Dust",
}

print(f"Rows loaded from source: {raw_rows}")
print(f"Rows used after forecast-target construction: {len(df)}")
print(f"Forecast horizon: {FORECAST_HORIZON}")
print(f"Engineered feature counts -> interact: {len(fe_interact)}, roll3: {len(fe_roll)}, poly2: {len(fe_poly)}")
print(f"Lag-weather features found: {len(lag_weather_cols)}")
print(f"Total modeling features (leakage-safe): {len(feature_cols)}")
for key, sdf in segments.items():
    print(f"{segment_labels[key]:<10s}: {len(sdf)} rows")

Rows loaded from source: 5069
Rows used after forecast-target construction: 4919
Forecast horizon: t_plus_1_week
Engineered feature counts -> interact: 28, roll3: 19, poly2: 15
Lag-weather features found: 36
Total modeling features (leakage-safe): 207
High Grown: 1392 rows
Low Grown : 1920 rows
Off-Grade : 842 rows
Dust      : 765 rows


In [3]:
"""Define models and adaptive time-series CV evaluation per segment."""

models = {
    "XGBoost": XGBRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        n_jobs=-1,
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=-1,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        random_state=SEED,
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
    ),
}

BASE_CV_SPLITS = 5
MIN_CV_SPLITS = 2
MIN_TEST_ROWS_PER_FOLD = 30
MIN_INITIAL_TRAIN_ROWS = 60

def choose_adaptive_splits(n_rows: int, base_splits: int = BASE_CV_SPLITS) -> int | None:
    """Choose fewer CV splits for small segments to keep train/test folds meaningful."""
    max_candidate = min(base_splits, n_rows - 1)
    for n_splits in range(max_candidate, MIN_CV_SPLITS - 1, -1):
        # For default TimeSeriesSplit: test_size ~= n_rows // (n_splits + 1)
        test_size = n_rows // (n_splits + 1)
        initial_train_size = n_rows - (n_splits * test_size)
        if test_size >= MIN_TEST_ROWS_PER_FOLD and initial_train_size >= MIN_INITIAL_TRAIN_ROWS:
            return n_splits

    # Fallback minimum only if technically feasible.
    if n_rows > MIN_CV_SPLITS:
        return MIN_CV_SPLITS
    return None

def compute_metrics(y_true, y_pred):
    """Compute RMSE, MAE, MAPE, and R2 on raw LKR prices."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

def evaluate_segment(segment_df, segment_name):
    """Run adaptive-fold TimeSeriesSplit for all models on one segment."""
    # Sort by auction order so each fold trains on past and tests on future only.
    sort_cols = [c for c in ["sale_year", "sale_number"] if c in segment_df.columns]
    if sort_cols:
        segment_df = segment_df.sort_values(sort_cols).reset_index(drop=True)

    X_seg = segment_df[feature_cols].copy()
    y_seg = segment_df[TARGET].copy()

    n_splits = choose_adaptive_splits(len(segment_df))
    if n_splits is None:
        raise ValueError(f"Not enough rows for time-series CV in segment {segment_name}: {len(segment_df)}")

    # Fold-safe imputation happens inside each fold through a pipeline.
    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_seg), start=1):
        X_train, X_test = X_seg.iloc[train_idx], X_seg.iloc[test_idx]
        y_train, y_test = y_seg.iloc[train_idx], y_seg.iloc[test_idx]

        for model_name, model in models.items():
            pipe = Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("model", clone(model)),
            ])
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)
            rmse, mae, mape, r2 = compute_metrics(y_test, preds)

            fold_rows.append(
                {
                    "Segment": segment_name,
                    "Fold": fold_idx,
                    "Model": model_name,
                    "RMSE": rmse,
                    "MAE": mae,
                    "MAPE": mape,
                    "R2": r2,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "n_splits_used": n_splits,
                }
            )

    fold_results = pd.DataFrame(fold_rows)
    summary = (
        fold_results.groupby(["Segment", "Model"], as_index=False)
        .agg(
            RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
            MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
            MAPE_mean=("MAPE", "mean"), MAPE_std=("MAPE", "std"),
            R2_mean=("R2", "mean"), R2_std=("R2", "std"),
            CV_Splits_Used=("n_splits_used", "max"),
        )
        .sort_values(["Segment", "RMSE_mean"])
        .reset_index(drop=True)
    )
    return fold_results, summary

In [4]:
"""Run segment-wise CV for all segments, show results, and save best-per-segment forecast table."""

all_fold_results = []
all_summary_results = []

for seg_key in ["high_grown", "low_grown", "off_grade", "dust"]:
    seg_df = segments[seg_key]
    seg_name = segment_labels[seg_key]

    if len(seg_df) < 30:
        print(f"Skipping {seg_name}: not enough rows for stable 5-fold CV.")
        continue

    fold_df, summary_df = evaluate_segment(seg_df, seg_name)
    all_fold_results.append(fold_df)
    all_summary_results.append(summary_df)
    print(f"Completed: {seg_name}")

cv_fold_results = pd.concat(all_fold_results, ignore_index=True)
cv_summary_results = pd.concat(all_summary_results, ignore_index=True)
cv_fold_results["Forecast_Horizon"] = FORECAST_HORIZON
cv_summary_results["Forecast_Horizon"] = FORECAST_HORIZON

print("\nSegment-wise 5-fold TimeSeries CV summary (sorted by RMSE):")
display(cv_summary_results.round(4))

print("\nBest model per segment (lowest RMSE_mean):")
best_per_segment = (
    cv_summary_results.sort_values(["Segment", "RMSE_mean"])
.groupby("Segment", as_index=False)
.first()[["Segment", "Model", "RMSE_mean", "MAE_mean", "MAPE_mean", "R2_mean", "Forecast_Horizon"]]
.sort_values("Segment")
.reset_index(drop=True)
.round(4)
)
display(best_per_segment)

tables_dir = Path("../reports/tables")
tables_dir.mkdir(parents=True, exist_ok=True)
best_path = tables_dir / "segmentwise_best_model_per_segment.csv"
best_per_segment.to_csv(best_path, index=False)
print(f"Saved: {best_path}")

Completed: High Grown
Completed: Low Grown
Completed: Off-Grade
Completed: Dust

Segment-wise 5-fold TimeSeries CV summary (sorted by RMSE):


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,CV_Splits_Used,Forecast_Horizon
0,High Grown,Random Forest,241.3588,28.6721,163.9178,15.4286,17.3939,3.4851,0.2324,0.1068,5,t_plus_1_week
1,High Grown,XGBoost,244.8622,26.8671,167.0627,14.7112,17.6366,3.8167,0.2078,0.1197,5,t_plus_1_week
2,High Grown,Gradient Boosting,246.5094,23.6387,169.4356,13.0100,17.7558,3.2166,0.1993,0.0834,5,t_plus_1_week
3,High Grown,LightGBM,250.1313,24.7180,172.9016,14.4944,18.1818,3.7245,0.1751,0.0953,5,t_plus_1_week
4,Low Grown,XGBoost,409.4064,72.6953,234.2346,21.9862,14.0736,0.8571,0.7205,0.0234,5,t_plus_1_week
5,Low Grown,Random Forest,411.4245,60.2093,238.3442,16.4007,14.4274,1.0983,0.7157,0.0167,5,t_plus_1_week
6,Low Grown,Gradient Boosting,440.9114,70.1987,271.4355,22.6418,16.9732,1.1070,0.6719,0.0497,5,t_plus_1_week
7,Low Grown,LightGBM,441.6481,100.6757,269.6616,51.4617,16.6398,2.2012,0.6742,0.0733,5,t_plus_1_week
8,Off-Grade,XGBoost,128.5175,19.2047,90.4546,14.5350,11.9850,3.3940,0.3286,0.1817,5,t_plus_1_week
9,Off-Grade,Random Forest,130.3194,17.6097,92.6785,15.2276,12.1821,3.6188,0.3125,0.1591,5,t_plus_1_week



Best model per segment (lowest RMSE_mean):


,Segment,Model,RMSE_mean,MAE_mean,MAPE_mean,R2_mean,Forecast_Horizon
0,Dust,Random Forest,164.9393,125.0351,13.9017,0.2927,t_plus_1_week
1,High Grown,Random Forest,241.3588,163.9178,17.3939,0.2324,t_plus_1_week
2,Low Grown,XGBoost,409.4064,234.2346,14.0736,0.7205,t_plus_1_week
3,Off-Grade,XGBoost,128.5175,90.4546,11.9850,0.3286,t_plus_1_week


Saved: ..\reports\tables\segmentwise_best_model_per_segment.csv


## Hyperparameter Tuning

This section applies the following hyperparameter tuning pipeline:
- 5-fold TimeSeriesSplit
- GridSearchCV
- tuning is done separately for each segment and for each model

In [5]:
"""Run segment-wise GridSearchCV for all models (no best-model filtering)."""

import json
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import clone

# Provide defaults in-cell so tuning runs even if helper defs were not executed earlier.
if "PARAM_GRIDS_ALL_MODELS" not in globals():
    PARAM_GRIDS_ALL_MODELS = {
        "Random Forest": {
            "model__n_estimators": [200, 300],
            "model__max_depth": [None, 10],
            "model__min_samples_split": [2, 5],
        },
        "Gradient Boosting": {
            "model__n_estimators": [200, 300],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [2, 3],
        },
        "XGBoost": {
            "model__n_estimators": [200, 300],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [4, 6],
        },
        "LightGBM": {
            "model__n_estimators": [200, 300],
            "model__learning_rate": [0.03, 0.05],
            "model__num_leaves": [31, 63],
        },
    }

def build_tuning_estimator(model_name, base_model):
    """Create leakage-safe estimator for GridSearchCV."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("model", clone(base_model)),
    ])

def evaluate_tuned_estimator_timeseries(sdf, estimator, n_splits=5):
    """Evaluate tuned estimator with chronological CV on one segment."""
    sort_cols = [c for c in ["sale_year", "sale_number"] if c in sdf.columns]
    if sort_cols:
        sdf = sdf.sort_values(sort_cols).reset_index(drop=True)

    X_seg = sdf[feature_cols].copy()
    y_seg = sdf[TARGET].copy()

    tscv = TimeSeriesSplit(n_splits=n_splits)
    rows = []
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_seg), start=1):
        X_train, X_test = X_seg.iloc[train_idx], X_seg.iloc[test_idx]
        y_train, y_test = y_seg.iloc[train_idx], y_seg.iloc[test_idx]

        est = clone(estimator)
        est.fit(X_train, y_train)
        preds = est.predict(X_test)
        rmse, mae, mape, r2 = compute_metrics(y_test, preds)

        rows.append({
            "Fold": fold_idx,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2,
        })
    return pd.DataFrame(rows)

inner_cv = TimeSeriesSplit(n_splits=5)

all_tuning_logs = []
all_tuned_metrics = []

for seg_name, sdf in segments.items():
    print(f"\nTuning segment: {seg_name} | rows={len(sdf)}")

    if len(sdf) < 20:
        print("  Skipped: not enough rows for stable 5-fold tuning.")
        continue

    X_seg = sdf[feature_cols].copy()
    y_seg = sdf[TARGET].copy()

    for model_name, base_model in models.items():
        if model_name not in PARAM_GRIDS_ALL_MODELS:
            print(f"  Skipped {model_name}: no parameter grid found.")
            continue

        estimator = build_tuning_estimator(model_name, base_model)
        param_grid = PARAM_GRIDS_ALL_MODELS[model_name]

        gscv = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring="neg_root_mean_squared_error",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
            return_train_score=False,
        )
        gscv.fit(X_seg, y_seg)

        # Keep full grid search logs for each segment and model.
        grid_log = pd.DataFrame(gscv.cv_results_)[
            ["params", "mean_test_score", "std_test_score", "rank_test_score"]
        ].copy()
        grid_log["Segment"] = seg_name
        grid_log["Model"] = model_name
        grid_log["mean_test_RMSE"] = -grid_log["mean_test_score"]
        all_tuning_logs.append(grid_log)

        # Evaluate tuned estimator again with 5-fold time-series splits.
        tuned_fold_eval = evaluate_tuned_estimator_timeseries(
            sdf=sdf,
            estimator=gscv.best_estimator_,
            n_splits=5,
        )

        all_tuned_metrics.append({
            "Segment": seg_name,
            "Model": model_name,
            "RMSE_mean": tuned_fold_eval["RMSE"].mean(),
            "RMSE_std": tuned_fold_eval["RMSE"].std(),
            "MAE_mean": tuned_fold_eval["MAE"].mean(),
            "MAE_std": tuned_fold_eval["MAE"].std(),
            "MAPE_mean": tuned_fold_eval["MAPE"].mean(),
            "MAPE_std": tuned_fold_eval["MAPE"].std(),
            "R2_mean": tuned_fold_eval["R2"].mean(),
            "R2_std": tuned_fold_eval["R2"].std(),
            "Selected_Hyperparameters": json.dumps(gscv.best_params_, sort_keys=True),
        })

        print(f"  Done: {model_name} | tuned RMSE={tuned_fold_eval['RMSE'].mean():.2f}")

segmentwise_tuning_logs = pd.concat(all_tuning_logs, ignore_index=True) if all_tuning_logs else pd.DataFrame()
segmentwise_tuned_summary = pd.DataFrame(all_tuned_metrics)

print("\nTuning logs shape:", segmentwise_tuning_logs.shape)
print("Tuned summary shape:", segmentwise_tuned_summary.shape)

display(
    segmentwise_tuned_summary
    .sort_values(["Segment", "RMSE_mean"])
    .reset_index(drop=True)
    .round(4)
)


Tuning segment: high_grown | rows=1392
  Done: XGBoost | tuned RMSE=242.70
  Done: LightGBM | tuned RMSE=245.56
  Done: Gradient Boosting | tuned RMSE=242.92
  Done: Random Forest | tuned RMSE=240.96

Tuning segment: low_grown | rows=1920
  Done: XGBoost | tuned RMSE=409.38
  Done: LightGBM | tuned RMSE=440.10
  Done: Gradient Boosting | tuned RMSE=435.64
  Done: Random Forest | tuned RMSE=410.47

Tuning segment: off_grade | rows=842
  Done: XGBoost | tuned RMSE=128.50
  Done: LightGBM | tuned RMSE=133.01
  Done: Gradient Boosting | tuned RMSE=138.37
  Done: Random Forest | tuned RMSE=129.64

Tuning segment: dust | rows=765
  Done: XGBoost | tuned RMSE=165.99
  Done: LightGBM | tuned RMSE=171.61
  Done: Gradient Boosting | tuned RMSE=170.55
  Done: Random Forest | tuned RMSE=164.88

Tuning logs shape: (128, 7)
Tuned summary shape: (16, 11)


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Selected_Hyperparameters
0,dust,Random Forest,164.8768,26.4467,124.9231,17.6115,13.8857,2.3650,0.2936,0.1964,"{""model__max_depth"": null, ""model__min_samples..."
1,dust,XGBoost,165.9919,26.2244,125.7615,19.7960,14.0338,2.5334,0.2858,0.1888,"{""model__learning_rate"": 0.03, ""model__max_dep..."
2,dust,Gradient Boosting,170.5450,24.3999,130.1334,17.0380,14.5871,2.6003,0.2492,0.1654,"{""model__learning_rate"": 0.03, ""model__max_dep..."
3,dust,LightGBM,171.6137,24.9632,129.7266,15.9681,14.5935,2.3324,0.2373,0.1834,"{""model__learning_rate"": 0.03, ""model__n_estim..."
4,high_grown,Random Forest,240.9595,28.5203,163.9967,15.5637,17.4093,3.4931,0.2349,0.1065,"{""model__max_depth"": null, ""model__min_samples..."
5,high_grown,XGBoost,242.6995,24.7885,164.5003,11.7520,17.3973,3.3884,0.2215,0.1134,"{""model__learning_rate"": 0.03, ""model__max_dep..."
6,high_grown,Gradient Boosting,242.9221,25.8779,165.9886,11.3823,17.4778,3.0769,0.2226,0.0917,"{""model__learning_rate"": 0.03, ""model__max_dep..."
7,high_grown,LightGBM,245.5629,23.4158,168.9560,13.4850,17.7788,3.5736,0.2025,0.1144,"{""model__learning_rate"": 0.03, ""model__n_estim..."
8,low_grown,XGBoost,409.3828,73.0131,234.5082,21.3770,14.0962,0.8593,0.7205,0.0237,"{""model__learning_rate"": 0.05, ""model__max_dep..."
9,low_grown,Random Forest,410.4706,61.5727,237.5256,16.5132,14.3465,1.0756,0.7172,0.0179,"{""model__max_depth"": null, ""model__min_samples..."


In [6]:
import os
import numpy as np
import pandas as pd

# Allow running this cell directly by loading prior outputs from disk when needed.
summary_csv = "../reports/tables/segmentwise_hyperparam_summary_all_models.csv"
logs_csv = "../reports/tables/_intermediate/segmentwise_hyperparam_gridsearch_all_results.csv"

if "segmentwise_tuned_summary" not in globals() or segmentwise_tuned_summary is None or segmentwise_tuned_summary.empty:
    if not os.path.exists(summary_csv):
        raise FileNotFoundError(
            f"Missing {summary_csv}. Run the tuning/save cells first or place the file in reports/tables."
        )
    segmentwise_tuned_summary = pd.read_csv(summary_csv)
    print(f"Loaded segmentwise_tuned_summary from: {summary_csv}")

if "segmentwise_tuning_logs" not in globals() or segmentwise_tuning_logs is None or segmentwise_tuning_logs.empty:
    if not os.path.exists(logs_csv):
        raise FileNotFoundError(
            f"Missing {logs_csv}. Run the tuning/save cells first or place the file in reports/tables."
        )
    segmentwise_tuning_logs = pd.read_csv(logs_csv)
    print(f"Loaded segmentwise_tuning_logs from: {logs_csv}")

# 1) Detailed model comparison table (all segment-model rows).
performance_table = segmentwise_tuned_summary.copy()
performance_table["RMSE_rank_in_segment"] = performance_table.groupby("Segment")["RMSE_mean"].rank(method="dense", ascending=True)
performance_table["R2_rank_in_segment"] = performance_table.groupby("Segment")["R2_mean"].rank(method="dense", ascending=False)
performance_table["RMSE_cv_pct"] = (performance_table["RMSE_std"] / performance_table["RMSE_mean"]) * 100.0
performance_table["R2_cv_pct"] = (performance_table["R2_std"].abs() / performance_table["R2_mean"].abs().replace(0, np.nan)) * 100.0

# 2) Grid-search breadth table (shows how much parameter space was explored for each model in each segment).
grid_coverage = (
    segmentwise_tuning_logs
    .groupby(["Segment", "Model"], as_index=False)
    .agg(
        Param_Combinations_Tried=("params", "count"),
        Grid_Best_CV_RMSE=("mean_test_RMSE", "min"),
        Grid_Worst_CV_RMSE=("mean_test_RMSE", "max"),
        Grid_CV_RMSE_STD_Avg=("std_test_score", "mean"),
    )
)

# 3) Final findings table: performance + tuning breadth in one place.
all_findings_table = (
    performance_table
    .merge(grid_coverage, on=["Segment", "Model"], how="left")
    .sort_values(["Segment", "RMSE_mean", "R2_mean"], ascending=[True, True, False])
    .reset_index(drop=True)
)

# 4) Reader-friendly combined snapshot for quick comparison.
segment_model_performance = all_findings_table[[
    "Segment",
    "Model",
    "RMSE_mean",
    "R2_mean",
    "RMSE_rank_in_segment",
    "R2_rank_in_segment",
    "RMSE_cv_pct",
    "R2_cv_pct",
    "Param_Combinations_Tried",
    "Grid_Best_CV_RMSE",
    "Grid_Worst_CV_RMSE",
    "Grid_CV_RMSE_STD_Avg",
]].copy()

print("All Findings Table (one row per segment and model):")
display(all_findings_table.round(4))

print("\nCombined RMSE/R2 performance table:")
display(segment_model_performance.round(4))

# Optional export so the same tables can go into report writing.
all_findings_table.to_csv("../reports/tables/segmentwise_all_findings_table.csv", index=False)
segment_model_performance.to_csv("../reports/tables/segmentwise_rmse_r2_combined.csv", index=False)

print("\nSaved:")
print("- ../reports/tables/segmentwise_all_findings_table.csv")
print("- ../reports/tables/segmentwise_rmse_r2_combined.csv")

All Findings Table (one row per segment and model):


,Segment,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,MAPE_mean,MAPE_std,R2_mean,R2_std,Selected_Hyperparameters,RMSE_rank_in_segment,R2_rank_in_segment,RMSE_cv_pct,R2_cv_pct,Param_Combinations_Tried,Grid_Best_CV_RMSE,Grid_Worst_CV_RMSE,Grid_CV_RMSE_STD_Avg
0,dust,Random Forest,164.8768,26.4467,124.9231,17.6115,13.8857,2.3650,0.2936,0.1964,"{""model__max_depth"": null, ""model__min_samples...",1.0,1.0,16.0403,66.8914,8,164.8768,165.4376,23.6328
1,dust,XGBoost,165.9919,26.2244,125.7615,19.7960,14.0338,2.5334,0.2858,0.1888,"{""model__learning_rate"": 0.03, ""model__max_dep...",2.0,2.0,15.7986,66.0490,8,166.3064,169.9922,23.8466
2,dust,Gradient Boosting,170.5450,24.3999,130.1334,17.0380,14.5871,2.6003,0.2492,0.1654,"{""model__learning_rate"": 0.03, ""model__max_dep...",3.0,3.0,14.3070,66.3912,8,170.5450,180.8947,21.3405
3,dust,LightGBM,171.6137,24.9632,129.7266,15.9681,14.5935,2.3324,0.2373,0.1834,"{""model__learning_rate"": 0.03, ""model__n_estim...",4.0,4.0,14.5462,77.2957,8,171.6137,172.0989,22.9473
4,high_grown,Random Forest,240.9595,28.5203,163.9967,15.5637,17.4093,3.4931,0.2349,0.1065,"{""model__max_depth"": null, ""model__min_samples...",1.0,1.0,11.8362,45.3587,8,240.9595,241.6495,25.2841
5,high_grown,XGBoost,242.6995,24.7885,164.5003,11.7520,17.3973,3.3884,0.2215,0.1134,"{""model__learning_rate"": 0.03, ""model__max_dep...",2.0,3.0,10.2137,51.2150,8,240.7466,247.0217,23.0782
6,high_grown,Gradient Boosting,242.9221,25.8779,165.9886,11.3823,17.4778,3.0769,0.2226,0.0917,"{""model__learning_rate"": 0.03, ""model__max_dep...",3.0,2.0,10.6528,41.1834,8,242.9221,247.7060,22.3581
7,high_grown,LightGBM,245.5629,23.4158,168.9560,13.4850,17.7788,3.5736,0.2025,0.1144,"{""model__learning_rate"": 0.03, ""model__n_estim...",4.0,4.0,9.5356,56.5004,8,245.5629,250.7659,22.0279
8,low_grown,XGBoost,409.3828,73.0131,234.5082,21.3770,14.0962,0.8593,0.7205,0.0237,"{""model__learning_rate"": 0.05, ""model__max_dep...",1.0,1.0,17.8349,3.2922,8,418.2528,435.9485,67.5302
9,low_grown,Random Forest,410.4706,61.5727,237.5256,16.5132,14.3465,1.0756,0.7172,0.0179,"{""model__max_depth"": null, ""model__min_samples...",2.0,2.0,15.0005,2.4917,8,410.4706,415.9801,59.1435



Combined RMSE/R2 performance table:


,Segment,Model,RMSE_mean,R2_mean,RMSE_rank_in_segment,R2_rank_in_segment,RMSE_cv_pct,R2_cv_pct,Param_Combinations_Tried,Grid_Best_CV_RMSE,Grid_Worst_CV_RMSE,Grid_CV_RMSE_STD_Avg
0,dust,Random Forest,164.8768,0.2936,1.0,1.0,16.0403,66.8914,8,164.8768,165.4376,23.6328
1,dust,XGBoost,165.9919,0.2858,2.0,2.0,15.7986,66.0490,8,166.3064,169.9922,23.8466
2,dust,Gradient Boosting,170.5450,0.2492,3.0,3.0,14.3070,66.3912,8,170.5450,180.8947,21.3405
3,dust,LightGBM,171.6137,0.2373,4.0,4.0,14.5462,77.2957,8,171.6137,172.0989,22.9473
4,high_grown,Random Forest,240.9595,0.2349,1.0,1.0,11.8362,45.3587,8,240.9595,241.6495,25.2841
5,high_grown,XGBoost,242.6995,0.2215,2.0,3.0,10.2137,51.2150,8,240.7466,247.0217,23.0782
6,high_grown,Gradient Boosting,242.9221,0.2226,3.0,2.0,10.6528,41.1834,8,242.9221,247.7060,22.3581
7,high_grown,LightGBM,245.5629,0.2025,4.0,4.0,9.5356,56.5004,8,245.5629,250.7659,22.0279
8,low_grown,XGBoost,409.3828,0.7205,1.0,1.0,17.8349,3.2922,8,418.2528,435.9485,67.5302
9,low_grown,Random Forest,410.4706,0.7172,2.0,2.0,15.0005,2.4917,8,410.4706,415.9801,59.1435



Saved:
- ../reports/tables/segmentwise_all_findings_table.csv
- ../reports/tables/segmentwise_rmse_r2_combined.csv
